# C3 Aerodynamics of airfoil in a 2D cascade

## Introduction

In the last session, the simulation of the NACA 65(1)-212 airfoil for 4 degrees of incidence angle was studied, with special focus on boundary layer resolution and grid convergence study. We adopt the medium mesh, and we study the behavior of the airfoil for several incidence angles, ranging from 0 to 12 degrees.

As a major upgrade, we introduce also the simulation of the **2D cascade**, as studied in the [`D2_mean_line_analysis_and_2d_cascade`](../1_Design/D2_mean_line_analysis_and_2d_cascade.ipynb) notebook. The 2D cascade simulated here is similar to the one from the S&P axial fan, but the airfoil profile 
is still our NACA.

## Learning objectives

1. **Simulate** the aerodynamics of an airfoil in a 2D cascade
2. **Obtain** average outlet velocity in the 2D cascade from simulation results
3. **Compute** Diffusion Factor and mechanical efficiency
4. **Discuss** the behavior of the airfoil for different flow rates

## Previous tasks
[] Study the [`D2_mean_line_analysis_and_2d_cascade`](../1_Design/D2_mean_line_analysis_and_2d_cascade.ipynb) notebook and refresh the basic concepts of mean line analysis

## Steps for the simulation of the 2D cascade

1. We make a copy of previous project and name it `NACA 65(1)-212 2D cascade`.
2. In this new project, delete all the runs and meshes. We are going to make a fresh start, using only the original geometry,
3. In `Simscale` it is possible to make some simple operations (as scale and rotate) on the CAD geometry. Scale the geometry, with a factor of 0.2 (chord will be $l = 60\,\text{mm}$), and rotate it, along the $z$ axis, the stagger angle $\xi = 50^\circ$. 
4. In the 2D cascade the $x$ axis is the axial direction of the fan, not the relative direction of flow. Also, in our case, the airfoil chord (300 mm) is huge for an axial fan. The first step, hence, is to scale and rotate
   ![CAD_transform](./images/C3_rotate_scale.png)
5. We are going to consider that the pitch is $s = 120 \,\text{mm}$ so that the solidity is $\sigma = 0.5$. The domain will be in the $y$ direction from $-s/2$ to $s/2$, end the boundary conditions in these places will be _periodic_, to simulate the presence of other blades in the $y$ direction.
6. Make an `Hex-dominant parametric` mesh with a discretization of 80x8x1 with a background box extending from (-0.3,-0.06,0) to (0.9,0.06,0.06). Check also that the `Material Point` is inside the background box, but outside the geometry.
7. To get a more accurate result around the airfoil and the wake, define a `Refinement box` primitive with bounding box from (-0.05,-0.06,0) to (0.15,0.06,0.06). 
8. The discretization of the main background mesh will be 100x10x1. The `Refinement box` level will be 2. Define also a boundary layer mesh around the airfoil with 2 layers, expansion ratio 1.2, and final layer thickness 0.5. The resulting mesh should be like this:
    ![Mesh](./images/C3_mesh.png)
    ![Mesh_detail](./images/C3_mesh_detail.png)
9. Define the material "Air" for the whole domain.
10. Define the boundary conditions:
    1.  The patches normal to $z$ direction are `symmetry`, like in the previous session with isolated airfoil
    2.  The airfoil is `wall`, with `wall function` for computation of boundary layer
    3.  The patch of minimum $x$ is `Velocity inlet` with velocity (10,13.76,0). This velocity will have an incidence angle of $4^\circ$, since the angle with respect the $x$ axis is $54^\circ$, 4 more than the stagger angle
    4.  The patch of maximum $x$ is `Pressure oulet` with an static pressure of $0 \,\text{Pa}$
    5.  The patches for minimum and maximum $y$ will be `Periodic`. That means that the solution obtained in both patches will be same, simulating a cycling domain.
11. Define monitors for forces and moments in the airfoil and the computation of $y^+$. Define a monitor to measure the average pressure in the inlet patch.
12. Run the simulation. Name it `alpha_4`, with reference to the incidence angle.

## Post-processing

1. We know the value of $\beta_1 = 54^\circ$ since it is an input. But the value of $\beta_2$ has to be measured. We are going to compute the value of $\beta_2$ from the distribution of $x$ and $y$ velocities downstream. Not too close to the trailing edge and neither too far away. Typically, about 20% to 50% of chord length is used. We are going to make this measurement about 20 mm downstream, 30% of chord length (showing the mesh can be helpful to pick the points), and save both velocities in `CSV` files. You can fins the files `C3_alpha_4_W2x.csv` and `C3_alpha_4_W2y.csv` in the `data` folder.
   ![plot over line](./images/C3_plot_over_line.png)
   ![velocity CSV](./images/C3_save_velocity_CSV.png)
   The angle $\beta_2$ has to be computed as mass weighted average of $x$ and $y$ velocities,
   $$ \tan(\beta_2) = \frac{ \overline{W_{2y}}} {\overline{W_{2x} } }  $$
   where 
   $$ \overline{W_{2x}} = \frac{\int{W_{2x}^2}\text{d}y }{ \int{W_{2x}}\text{d}y }$$
   $$ \overline{W_{2y}} = \frac{\int{W_{2y} W_{2x} }\text{d}y }{ \int{W_{2x}}\text{d}y }$$
   

In [1]:
import pandas as pd
from resources.utils import compute_bulk_beta2_from_distribution

# Read data for X and Y velocities from CSV files
df_x = pd.read_csv('data/C3_alpha_4_W2x.csv')
df_y = pd.read_csv('data/C3_alpha_4_W2y.csv')

# Extract relevant columns from the dataframes
y_points = df_x['Distance along the path (m)'].values
W2x_points = df_x['Velocity X (m/s)'].values
W2y_points = df_y['Velocity Y (m/s)'].values

# Compute bulk beta2 from the distribution of velocities
beta2, W2x_mean, W2y_mean = compute_bulk_beta2_from_distribution(y_points, W2x_points, W2y_points)

Computed bulk mass-weighted outlet flow angle: beta2 = 50.21 degrees
Computed bulk mass-weighted axial velocity: W2x_mass = 10.01 m/s
Computed bulk mass-weighted tangential velocity: W2y_mass = 12.01 m/s


2. With the value of $\beta_2$ the aerodynamic coefficients $C_L$ and $C_D$, can be estimated. Forces, that are monitored and saved as explained in the [C2 notebook](./C2_boundary_layer_and_GCI.ipynb), are needed to make this computation

In [2]:
from resources.utils import get_forces_from_csv

F_x, F_y, F_x_err, F_y_err = get_forces_from_csv('data/C3_alpha_4_forces.csv')


Mean forces: Fx = -0.17 N, Fy = 0.15 N
Standard deviation of forces: Fx_err = 3.1e-08 N, Fy_err = 2.4e-08 N


3. The main characteristics of the cascade can now be computed. Take a look at the function `compute_cascade_coefficients` in the [utils.py](./resources/utils.py) file to check how are they computed using the usual expressions in the book by Lewis.

In [3]:
import pprint # to print the dictionary in a readable format
from resources.utils import compute_cascade_coefficients

c_x = 10 # m/s
U_y1 = 13.76  # m/s
rho =1.2 # kg/m^3
s = 0.12 # pitch (m)
l = 0.06 # chord (m)
b = 0.06 # width (m)

cascade_coefficients = compute_cascade_coefficients(F_x=F_x, F_y=F_y, c_x=c_x, U_y1=U_y1, rho=rho, s=s, l=l, b=b, beta_2=beta2)
pprint.pprint(cascade_coefficients)

{'C_D': np.float64(0.02387393544021181),
 'C_L': np.float64(0.4036569875449257),
 'DF': np.float64(0.1845168742526736),
 'D_force': np.float64(0.013715623132878094),
 'L_force': np.float64(0.23190173777524248),
 'W_m': np.float64(16.308682900826394),
 'beta_1_deg': np.float64(53.99243843274894),
 'beta_2_deg': np.float64(50.20877782174788),
 'beta_m_deg': np.float64(52.180908935446396)}


Note, on one hand, that the value of $C_L$ is lower than that computed for an isolated airfoil (C2 notebook) and, on the other hand, that the value of DF is much lower than 0.5, which indicates that this airfoil is in a comfortable performance zone, far away from stall (DF ~ 0.5).

4. Efficiency of this 2D cascade can be estimated computing hydraulic power as usual,
   $$ P_H = P Q$$
   and mechanical power with tangential force and velocity
   $$ P_m = U_y F_y $$
   Note the the pressure (negative) is obtained from the average value in the inlet patch in the simulation

In [5]:
Q = c_x * b * s
P = 23.77 # Inlet negative pressure, from simulation data (Pa)
P_H = P * Q # Hydraulic power (W)
print(f"Hydraulic power: {P_H:.2f} W")
P_m = U_y1 * F_y # Mechanical power (W)
print(f"Mechanical power: {P_m:.2f} W")
eta = P_H / P_m # Efficiency
print(f"Efficiency: {eta:.4f}")

Hydraulic power: 1.71 W
Mechanical power: 2.11 W
Efficiency: 0.8128


## Some questions

1. Why is $C_L$ lower in the 2D cascade than in the isolated airfoil?
   
   Answer:
   
   ***
2. The velocity $W_{y2}$, needed to get the angle $\beta_2$, has been computed from the velocity profile from the simulation. Another method can be using momentum balance in the $y$-direction
   $$ F_y = \rho \dot{m} \left( W_{y1} - W{y2}\right)$$
   Use this expression to compute $y$ velocity in outlet and $\beta_2$ and compare results. Which one do you think is more reliable? 
   Answer:

   ***
3. Compare the pressure obtained by the simulation with the pressure rise predicted by Euler's equation
   $$ \Delta p = \rho U \left( C_{\theta 2} - C_{\theta 1} \right)$$
   Remember that $\mathbf{C}$ is the **absolute velocity** of air, and the simulation has been made in relative reference frame.

   Answer:

   ***

## Tasks

Make the same analysis, computing specially $DF$ and $\eta$, for several flow rates. Use the $c_x$ velocities to get an incidence angle in the range from -4 to 12 with steps of 4 degrees (that is, [-4,0,4 (already done),8,12]). Discuss the results.

## Your project

Make the numerical estimation of performance and efficiency of your axial fan according to the mean line analysis performed in the Design part.
Compare with analytical estimation 